# 02 — Validation (DCC process **V1**)

**SEA-FORWARD** · OPERA Capacity Development · OceanPrediction-A toolkit

This notebook is the **manual / visual half** of Step 4.2 in the SEA-FORWARD
operational workflow. The automated half, `sftools/run_validation.py`
(Step 4.1), writes `validation_report.json`/`.txt` and `taylor_diagram.png`
per cycle -- both halves now go through the exact same statistics engine,
`sftools.validation_godae` (see Section 4), so they can never silently
disagree. This notebook implements Component **Validation Module** (DCC
process **V1**) as a Jupyter Notebook: bias maps, scatter plots, Taylor
diagrams, time series, and (Section 7) optional in-situ / class-4 scoring.

### Where V1 sits in the OceanPrediction-A chain

```
 U2 (atmospheric forcing) ---+
 U4 (ocean boundary cond.) --+---> C1 (CROCO ocean model) ---> V1 (this notebook) ---> D1 (01/03/04 notebooks)
 U5 (bathymetry) ------------+
```

**V1 -- Validation & Model Intercomparison**: statistical comparison of model
output against a bundled reference (an assimilative reanalysis such as
GLORYS12V1, and/or in-situ observations) -- RMSE, bias, spatial correlation,
Taylor-diagram skill metrics. A model that fails V1 should not be trusted
for the downstream D1 applications (visualisation, exercises, sensitivity
analysis) in the other three notebooks.

> **OceanPrediction-A** is the *simplest* DCC Architecture blueprint: a
> single deterministic run, no data assimilation. That's exactly why V1
> matters here -- with no assimilation step correcting the model against
> observations as it runs, all of the quality control happens *after* the
> fact, in this notebook and in `run_validation.py`.

**Reference dataset.** The reference used below is the parent product CROCO
was downscaled from (GLORYS12V1 reanalysis for a hindcast) -- this is the
"Reference Results Dataset" (D10.3), DOI-referenced on Zenodo. If in-situ
observations (CMEMS In-Situ TAC, https://doi.org/10.48670/moi-00036) are
also available (Section 7), `sftools.validation_godae.validate_against_insitu`
extends the same statistics to point/profile comparisons, per GODAE depth
layer -- see that module's docstring.

*Docstrings and this markdown are in English per FR-09; French translation
of the user-facing narrative is coordinated separately with the
documentation team -- see the closing cell.*


In [ ]:
# ----------------------------------------------------------------------
# Setup -- run from the notebooks/ folder so sftools imports (see sftools/README.md)
# ----------------------------------------------------------------------
import sys, os
sys.path.insert(0, "..")   # repo root, so `import sftools...` resolves

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sftools.postprocess as pp
import sftools.validation as val
import sftools.validation_godae as vg
import sftools.plotting as pl

import _demo_data

CROCO_HIS, REFERENCE, YORIG, IS_DEMO = _demo_data.get_paths()

if IS_DEMO:
    print("!! DEMO DATA !! Real D10.2/D10.3 files were not found at the configured")
    print("   paths, so a small synthetic stand-in was generated instead, so this")
    print("   notebook can still run end-to-end (see notebooks/_demo_data.py).")
    print("   Numbers below are illustrative only -- NOT a validated model result.")
else:
    print("Using real project data:")
print(f"  CROCO history : {CROCO_HIS}")
print(f"  Reference     : {REFERENCE}")


## 1. Load model output and check the run

Before comparing anything, a quick look at what actually came out of C1
(the CROCO run): grid size, time coverage, and Forecasting Accuracy
Validation Criterion **"Numerical stability -- No NaN or overflow in any
output field"** (see Technical Specification Section 9.3).


In [ ]:
ds = pp.open_history(CROCO_HIS, Yorig=YORIG)

print(f"grid          : {ds.sizes['eta_rho']} x {ds.sizes['xi_rho']}  "
      f"({ds.sizes.get('s_rho', '?')} sigma levels)")
print(f"time steps    : {ds.sizes['time']}")
print(f"time coverage : {pp.times(ds)[0]}  ->  {pp.times(ds)[-1]}")

stability_ok = True
for var in ('zeta', 'temp', 'salt', 'u', 'v'):
    vals = ds[var].values
    n_nan = int(np.isnan(vals).sum())
    n_inf = int(np.isinf(vals).sum())
    ok = (n_nan == 0) and (n_inf == 0)
    stability_ok &= ok
    print(f"  {var:5s}: {'OK  ' if ok else 'FAIL'}  (NaN={n_nan}, Inf={n_inf})")

print()
print(f"Numerical stability (FR pass criterion): {'PASS' if stability_ok else 'FAIL'}")


## 2. Bias maps -- CROCO vs reference

Three-panel maps (CROCO | reference | difference) plus domain-averaged
statistics, for the three headline Forecasting Accuracy Validation
Criteria variables (SST, SSH, surface currents) using
`sftools.validation` (see that module's docstring for the regridding
method).


In [ ]:
_, sst_stats = val.compare_sst(CROCO_HIS, REFERENCE, Yorig=YORIG)
# pass/fail against this and every other Section 9.3 criterion is decided
# once, from the shared GODAE scorecard, in Section 6 below.


In [ ]:
_, ssh_stats = val.compare_ssh(CROCO_HIS, REFERENCE, Yorig=YORIG)


In [ ]:
_, cur_stats = val.compare_currents(CROCO_HIS, REFERENCE, Yorig=YORIG)
print()
print("Surface velocities pass criterion is QUALITATIVE (visual consistency with")
print("expected gyre/coastal-jet circulation) -- inspect the vector map above.")


In [ ]:
plon, plat, pfield = val.load_parent(REFERENCE, 'salt')
salt_on_croco = val.regrid_to_croco(plon, plat, pfield, ds)
salt_stats = val.domain_statistics(pp.surface(ds, 'salt').values, salt_on_croco)
val._print_stats('SSS', salt_stats)


## 3. Scatter plots -- pointwise CROCO vs reference

The bias maps above show *where* the differences are; a scatter plot of
every grid point (CROCO value vs. co-located reference value) shows the
overall *shape* of the agreement -- a tight cloud along the 1:1 line means
good agreement; a cloud offset from the line indicates a systematic bias;
a fan-shaped cloud indicates the CROCO field is over/under-dispersed
relative to the reference (this is exactly what the Taylor diagram below
summarises in one number: the model/reference standard-deviation ratio).


In [ ]:
def scatter_vs_reference(model_field, ref_field, label, units, ax=None, max_points=20000):
    '''Pointwise CROCO-vs-reference scatter with a 1:1 line and the
    domain_statistics() bias/RMSE/corr annotated on the plot.'''
    m = np.asarray(model_field).ravel()
    r = np.asarray(ref_field).ravel()
    ok = np.isfinite(m) & np.isfinite(r)
    m, r = m[ok], r[ok]
    if len(m) > max_points:                      # subsample for a readable/fast plot
        idx = np.random.default_rng(0).choice(len(m), max_points, replace=False)
        m, r = m[idx], r[idx]

    s = val.domain_statistics(m, r)
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    lo, hi = np.nanpercentile(np.concatenate([m, r]), [1, 99])
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='1:1')
    ax.scatter(r, m, s=4, alpha=0.25, color='C0')
    ax.set_xlabel(f'reference {label} ({units})')
    ax.set_ylabel(f'CROCO {label} ({units})')
    ax.set_title(f"{label}: bias={s['bias']:+.2f}  RMSE={s['rmse']:.2f}  corr={s['corr']:.2f}")
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_aspect('equal', adjustable='box')
    ax.legend()
    return ax


fig, axes = plt.subplots(1, 2, figsize=(11, 5))

sst = pp.surface(ds, 'temp').values
plon, plat, pfield = val.load_parent(REFERENCE, 'temp')
sst_ref = val.regrid_to_croco(plon, plat, pfield, ds)
scatter_vs_reference(sst, sst_ref, 'SST', 'degC', ax=axes[0])

ssh = ds['zeta'].isel(time=-1).values
plon, plat, pfield = val.load_parent(REFERENCE, 'ssh')
ssh_ref = val.regrid_to_croco(plon, plat, pfield, ds)
scatter_vs_reference(ssh, ssh_ref, 'SSH', 'm', ax=axes[1])

fig.tight_layout()


## 4. Taylor diagram

A **Taylor diagram** summarises three skill statistics in a single polar
plot: the correlation with the reference (angle), the ratio of the model's
standard deviation to the reference's (radial distance), and -- implicitly,
via distance from the reference point -- the centred RMSE. It's the
standard GODAE OceanView / CMEMS intercomparison summary plot, and is what
`sftools.validation_godae` (module `vg` here) was built to produce; see
that module's docstring for the full GODAE metric set (bias, RMSD, unbiased
RMSD, correlation, two scatter-index variants, std-ratio).

We score SST, SSH, SSS and surface current speed against the reference in
one pass and put them all on the same diagram, so a single glance shows
which variable(s) are driving any V1 concern.


In [ ]:
# All four variables now go through the same call -- SSH's arbitrary-
# geoid-reference issue (CROCO zeta has no absolute reference level, so a
# raw-level comparison would score that mismatch as spurious bias) is
# handled inside godae_scorecard_croco_vs_glorys itself now (ssh_anomaly=True
# by default), so no special-casing is needed here any more.
rows = []
for var in ('temp', 'ssh', 'salt', 'speed'):
    s = vg.godae_scorecard_croco_vs_glorys(CROCO_HIS, REFERENCE, var, Yorig=YORIG)
    rows.append({**s, 'vs': 'reference', 'layer': 'all'})

report = pd.DataFrame(rows)
vg.print_scorecard_table(report)


In [ ]:
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, polar=True)
has_neg = bool((report['corr'] < 0).any())
thetamax = 180 if has_neg else 90
corr_ticks = ([-1.0, -0.5, 0, 0.5, 0.8, 0.9, 0.95, 0.99, 1.0] if has_neg
              else [0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99, 1.0])
ax.set_thetamin(0); ax.set_thetamax(thetamax)
ax.set_xticks(np.arccos(corr_ticks)); ax.set_xticklabels([str(c) for c in corr_ticks])
ax.set_ylabel('normalised std dev (model / reference)')
finite = report['std_ratio'][np.isfinite(report['std_ratio'])]
ax.set_ylim(0, max(1.6, finite.max() * 1.2) if len(finite) else 1.6)
ax.plot(0, 1, 'k*', ms=16, label='reference')
for _, row in report.iterrows():
    theta = np.arccos(np.clip(row['corr'], -1, 1))
    ax.plot(theta, row['std_ratio'], 'o', ms=10,
           label=f"{row['variable']}  (RMSD={row['rmsd']:.2f})")
ax.set_title('Taylor diagram -- CROCO vs reference', pad=20)
ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1.0), fontsize=9)
fig.tight_layout()


## 5. Time series comparison

A single-point time series makes the *temporal* behaviour visible in a way
maps can't -- useful for checking that CROCO isn't drifting away from the
reference over the run (a common failure mode in a hindcast with weak/no
nudging). Picked here at a coastal point, since that's also where the
upwelling exercise in `03_exercises.ipynb` focuses.


In [ ]:
clon, clat, cmask = pp.lonlatmask(ds)
# a coastal-ish point: 75% of the way across the grid, mid-latitude
j0 = ds.sizes['eta_rho'] // 2
i0 = int(0.75 * ds.sizes['xi_rho'])
lon0, lat0 = float(clon[j0, i0]), float(clat[j0, i0])
print(f'time series point: ({lon0:.2f}, {lat0:.2f})')

val.compare_timeseries(CROCO_HIS, REFERENCE, 'temp', lon0, lat0, Yorig=YORIG)


## 6. Automated pass/fail summary (V1)

Pulling together every criterion from the Forecasting Accuracy Validation
Criteria table (Technical Specification Section 9.3) into one summary.
This is the notebook-side mirror of what `run_validation.py` (Step 4.1,
automated) writes to `validation_report.html` -- the two should always
agree; if they don't, that's worth reporting (see the QA note in the
closing cell).


In [ ]:
# Sourced from the SAME GODAE scorecard (report, built in Section 4) that
# sftools/run_validation.py uses for the automated per-cycle report -- so
# this interactive summary and the batch/cron reports can never silently
# disagree (see that script's docstring).
by_var = {r['variable']: r for r in rows}

criteria = [
    ('SST domain-avg RMSD < 0.5 degC',       by_var['temp']['rmsd'] < 0.5,  f"{by_var['temp']['rmsd']:.3f} degC"),
    ('SSH spatial correlation > 0.90',       by_var['ssh']['corr'] > 0.90, f"{by_var['ssh']['corr']:.3f}"),
    ('Salinity domain-avg |bias| < 0.2 PSU', abs(by_var['salt']['bias']) < 0.2, f"{by_var['salt']['bias']:+.3f} PSU"),
    ('Numerical stability (no NaN/Inf)',     stability_ok, 'see section 1'),
]

print(f"{'Criterion':45s} {'Result':6s}  Value")
print('-' * 70)
all_pass = True
for name, passed, value in criteria:
    all_pass &= passed
    print(f"{name:45s} {'PASS' if passed else 'FAIL':6s}  {value}")
print('-' * 70)
print()
print(f"Overall V1 status: {'PASS' if all_pass else 'FAIL -- see failing criteria above'}")
if IS_DEMO:
    print('(DEMO DATA -- this PASS/FAIL is illustrative only, not a validated result)')
print()
print("Note: surface-current skill (by_var['speed']) has no fixed Section 9.3")
print("threshold -- its criterion is qualitative (inspect the vector map in")
print("Section 2), same as sftools/run_validation.py.")

ds.close()


## 7. In-situ validation (optional, class 4)

Everything above is "class 1/2" (model vs. an assimilative reanalysis/
reference product) -- the standard GODAE OceanView taxonomy also defines
"class 4": model vs. independent in-situ observations, which `sftools.
validation_godae.validate_against_insitu()` scores per GODAE depth layer
against CMEMS in-situ TAC profiles (https://doi.org/10.48670/moi-00036).

This is optional here (as it is in `sftools/run_validation.py` via
`--insitu-files`/`--require-insitu-pass`) because in-situ coverage for a
given cycle/region is often patchy -- a cycle with zero nearby profiles
isn't a validation failure, just a cycle with nothing to check here.


In [ ]:
# Set to a real glob pattern to enable, e.g.:
#   INSITU_FILES = "../hindcast/downloaded_data/INSITU/2025-12-2*.nc"
INSITU_FILES = None

if INSITU_FILES:
    ds2 = pp.open_history(CROCO_HIS, Yorig=YORIG)   # reopened: the Section 6 cell closed ds
    insitu_report = vg.validate_against_insitu(
        CROCO_HIS, INSITU_FILES, glorys_file=REFERENCE,
        variables=("temp", "salt"), Yorig=YORIG)
    ds2.close()
    if insitu_report.empty:
        print("No in-situ observations matched this cycle/region -- nothing to score.")
    else:
        vg.print_scorecard_table(insitu_report)
        # a quick surface-layer check, mirroring run_validation.py's
        # --require-insitu-pass threshold (informational here either way)
        surf = insitu_report[(insitu_report["variable"] == "temp")
                             & (insitu_report["vs"] == "CROCO")
                             & (insitu_report["layer"] == "surface (0-10 m)")]
        if len(surf):
            bias = float(surf.iloc[0]["bias"])
            print()
            print(f"In-situ surface temp bias: {bias:+.3f} degC "
                 f"({'within' if abs(bias) < 1.0 else 'OUTSIDE'} the +/-1.0 degC "
                 f"informational tolerance)")
else:
    print("INSITU_FILES not set -- skipping in-situ validation for this run.")
    print("(This is expected/fine; see the markdown above.)")


## 8. Batch validation across every forecast cycle

Everything above validates a *single* cycle, loaded interactively. In
production, every forecast cycle gets its own dated directory (e.g.
`forecast/model-runs/Canary_12/20260714/`) -- `forecast/validate_all_cycles.sh`
automates Step 4.1 across all of them: it discovers every cycle directory
that has a `croco_his.nc`, validates any not already validated (writing
`validation_report.json`/`.txt` next to that cycle's output, and one row to
a running `validation_summary.csv` per region), and is safe to run
repeatedly -- see `forecast/install_validation_crontab.sh` and the
`notebooks/README.md` for scheduling it with cron so new cycles get
validated automatically as they land.

This section can (a) trigger a batch run directly from the notebook, and
(b) load the accumulated `validation_summary.csv` to show trends across
cycles -- catching, e.g., a slow drift in SST RMSE that a single cycle's
V1 check wouldn't reveal on its own.


In [ ]:
import subprocess

# Off by default -- this can take a while across many cycles and isn't
# something you want an accidental "Run All" to kick off unattended.
# Flip to True to trigger a batch run from here instead of a shell/cron.
RUN_BATCH_VALIDATION = False
BATCH_REGION = None   # e.g. "Canary_12"; None validates every region found

if RUN_BATCH_VALIDATION:
    cmd = ["../forecast/validate_all_cycles.sh"]
    if BATCH_REGION:
        cmd += ["--region", BATCH_REGION]
    print(f"running: {' '.join(cmd)}")
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout[-3000:])
    if proc.returncode != 0:
        print(f"exit code {proc.returncode} -- at least one cycle failed or errored; "
             "see forecast/logs/ for the full log")
else:
    print("RUN_BATCH_VALIDATION is False -- skipping. Set it to True to run a batch "
         "validation pass from here, or run forecast/validate_all_cycles.sh directly "
         "from a shell (recommended for a real production batch -- see notebooks/README.md).")


In [ ]:
import glob

SUMMARY_GLOB = "../forecast/model-runs/*/validation_summary.csv"
summary_files = sorted(glob.glob(SUMMARY_GLOB))

if not summary_files:
    print("No validation_summary.csv found yet under forecast/model-runs/*/.")
    print("Run forecast/validate_all_cycles.sh at least once (see the cell above, "
         "or from a shell) to generate batch-validation history to plot here.")
    if IS_DEMO:
        print()
        print("(Expected in demo mode: the synthetic demo data is a single stand-in "
             "file, not a real multi-cycle archive -- there's nothing honest to batch "
             "here yet. This section is fully functional once real cycles exist.)")
    history = None
else:
    history = pd.concat([pd.read_csv(f) for f in summary_files], ignore_index=True)
    history["validated_at_utc"] = pd.to_datetime(history["validated_at_utc"], errors="coerce")
    history = history.sort_values("validated_at_utc")
    print(f"loaded {len(history)} validation records from {len(summary_files)} region(s)")
    display(history.tail(10))


In [ ]:
if history is not None and len(history) > 0:
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
    for region, grp in history.groupby("region"):
        axes[0].plot(grp["validated_at_utc"], grp["sst_rmse"], "o-", label=region)
        axes[1].plot(grp["validated_at_utc"], grp["ssh_corr"], "o-", label=region)
        axes[2].plot(grp["validated_at_utc"], grp["salt_bias"], "o-", label=region)

    axes[0].axhline(0.5, color="r", ls="--", lw=1, label="pass threshold")
    axes[0].set_ylabel("SST RMSE (degC)")
    axes[1].axhline(0.90, color="r", ls="--", lw=1, label="pass threshold")
    axes[1].set_ylabel("SSH corr")
    axes[2].axhline(0.2, color="r", ls="--", lw=1)
    axes[2].axhline(-0.2, color="r", ls="--", lw=1, label="pass threshold")
    axes[2].set_ylabel("Salinity bias (PSU)")
    axes[2].set_xlabel("cycle validated_at (UTC)")
    for ax in axes:
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    fig.suptitle("V1 criteria across all validated forecast cycles")
    fig.tight_layout()
    plt.show()

    n_fail = int((~history["all_pass"].astype(bool)).sum())
    print(f"{n_fail} of {len(history)} validated cycles FAILED at least one criterion.")


---
## Notes

- **DCC linkage (FR-12):** this notebook implements process **V1**
  (Verification & Analysis layer). Its inputs come from **C1** (the CROCO
  run) and its outputs (validated CROCO fields, this pass/fail summary)
  feed the **D1** notebooks (`01_visualisation.ipynb`,
  `03_exercises.ipynb`, `04_sensitivity.ipynb`, `05_animation.ipynb`).
- **Bilingual documentation (FR-09):** this notebook's markdown and
  docstrings are authored in English. French translation of the
  user-facing narrative text is coordinated with the documentation team
  as a separate pass (translating in place risks drifting out of sync
  with the code -- the English version here is the source of truth).
- **QA (Testing and Validation Plan Section 9.1):** this notebook is
  designed to execute without errors from a fresh kernel restart +
  run-all, using the demo-data fallback in `_demo_data.py` if real
  D10.2/D10.3 data isn't present locally yet.
- **Automation:** Section 8 above covers batch validation across every forecast cycle via `forecast/validate_all_cycles.sh` and scheduling it with `forecast/install_validation_crontab.sh` -- see `notebooks/README.md`.
- **Next:** `03_exercises.ipynb` for guided, hands-on diagnostics
  (upwelling index, mixed-layer depth, coastal jet, eddy detection) built
  on the same CROCO output.
